# 12 K-Means 聚类

依赖安装说明：`pip install numpy matplotlib scikit-learn`

K-Means 是无监督学习模型，用来把样本分成 K 个簇。它没有标签，目标是让每个点离自己的簇中心尽量近。


## 1. 数学逻辑

K-Means 最小化簇内平方距离：

$$\min_{C,\mu}\sum_{k=1}^{K}\sum_{x_i\in C_k}||x_i-\mu_k||^2$$

算法循环两步：

1. 分配：每个点归到最近的中心。
2. 更新：每个中心移动到自己簇内点的均值。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(42)
X, _ = make_blobs(n_samples=260, centers=3, cluster_std=0.7, random_state=42)
plt.scatter(X[:,0], X[:,1], s=24)
plt.title('没有标签的点云')
plt.show()


In [ ]:
# 从零实现 K-Means
K = 3
rng = np.random.default_rng(42)
centers = X[rng.choice(len(X), size=K, replace=False)]

for step in range(15):
    distances = np.sqrt(((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2))
    labels = distances.argmin(axis=1)
    new_centers = np.array([X[labels == k].mean(axis=0) for k in range(K)])
    shift = np.sqrt(((new_centers - centers) ** 2).sum())
    centers = new_centers
    print(f'step {step+1:2d} | center shift={shift:.4f}')
    if shift < 1e-4:
        break

plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=24)
plt.scatter(centers[:,0], centers[:,1], c='red', marker='x', s=120)
plt.title('从零 K-Means 结果')
plt.show()


In [ ]:
model = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = model.fit_predict(X)
print('inertia:', round(model.inertia_, 3))
print('silhouette:', round(silhouette_score(X, labels), 3))

plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=24)
plt.scatter(model.cluster_centers_[:,0], model.cluster_centers_[:,1], c='red', marker='x', s=120)
plt.title('sklearn KMeans')
plt.show()


## 2. 常见误区

- K-Means 需要预先指定 K。
- 它偏好球形、大小接近的簇；对月牙形、密度不均数据效果差。
- 初始化会影响结果，所以通常多次初始化。

## 3. 小实验

- 把 `K` 改成 2 或 4。
- 增大 `cluster_std`，观察簇重叠。
- 换成 `make_moons`，看 K-Means 的局限。
